# Chapter 10: Deep GA on a Colab GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcellmajor/Chapter10_updated/blob/main/colab/deep_ga_atari.ipynb)

This notebook trains the PyTorch port of Chapter 10 from
[marcellmajor/Chapter10_updated](https://github.com/marcellmajor/Chapter10_updated).
The [README](https://github.com/marcellmajor/Chapter10_updated#readme) explains what the
experiment does, resource estimates, and resume restrictions.

Open **Runtime → Change runtime type → GPU**, then run cells individually, in order.
The repository is public, so no GitHub sign-in or token is needed.

Code is cloned into the VM's local disk. Checkpoints, diagnostics, and videos go to
Google Drive. Reconnect by rerunning setup with the same run name and training budget.

CPU behavior has been tested locally. This notebook is the procedure for validating
CUDA on your actual Colab runtime; GPU speed and learning performance are not yet measured.


In [ ]:
REPO_URL = "https://github.com/marcellmajor/Chapter10_updated.git"
RUN_NAME = "frostbite-v1"
NUM_ENVS = 64                 # Try 32 or 16 if GPU memory is insufficient.
TOTAL_STEPS = 1_000_000        # Total budget, including steps already completed.
RUN_PRETRAINED = False         # Optional three-episode original-policy replay.


## 1. Mount Drive and fetch the code

This cell mounts Google Drive and clones the `main` branch of
[marcellmajor/Chapter10_updated](https://github.com/marcellmajor/Chapter10_updated) into
`/content/Chapter10_updated`. Rerunning it on the same VM fast-forwards that checkout to
the latest pushed commit; a new VM clones again. Results are kept in
`MyDrive/chapter10-runs/<RUN_NAME>/`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')

SOURCE_DIR = Path('/content/Chapter10_updated')
if not (SOURCE_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(SOURCE_DIR)], check=True)
else:
    origin = subprocess.check_output(['git', '-C', str(SOURCE_DIR), 'remote', 'get-url', 'origin'], text=True).strip()
    if origin != REPO_URL:
        raise ValueError(f'{SOURCE_DIR} is a checkout of {origin}, not {REPO_URL}. Use a new VM.')
    # Fails instead of discarding any edits made to the checkout on this VM.
    subprocess.run(['git', '-C', str(SOURCE_DIR), 'pull', '--ff-only'], check=True)
print(subprocess.check_output(['git', '-C', str(SOURCE_DIR), 'log', '-1', '--oneline'], text=True))

assert (SOURCE_DIR / 'requirements.txt').is_file(), SOURCE_DIR
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in {'.', '..'}, 'Use a simple run name.'
os.chdir(SOURCE_DIR)
RUNS_DIR = Path('/content/drive/MyDrive/chapter10-runs')
OUT_DIR = RUNS_DIR / RUN_NAME
DIAGNOSTICS_DIR = OUT_DIR / 'diagnostics'
DIAGNOSTICS_DIR.mkdir(parents=True, exist_ok=True)
print('Source:', SOURCE_DIR)
print('Persistent output:', OUT_DIR)


## 2. Install dependencies

Retain Colab's preinstalled CUDA torch if it meets the requirements. If installation
requests a runtime restart, restart and rerun setup before continuing.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


## 3. Verify the GPU and save diagnostics

This check must succeed before testing or training. Diagnostics record the code revision,
package versions, GPU, and CPU count so a later error can be reproduced.


In [ ]:
import json
import platform
import ale_py
import gymnasium
import numpy as np
import torch

assert torch.cuda.is_available(), 'No CUDA GPU available. Select a GPU runtime and reconnect.'
# Also check that the installed CUDA wheel can execute a GPU operation.
assert torch.ones(1, device='cuda').sum().item() == 1
revision = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True)
info = {
    'python': sys.version, 'platform': platform.platform(), 'cpu_count': os.cpu_count(),
    'torch': torch.__version__, 'cuda_build': torch.version.cuda,
    'gymnasium': gymnasium.__version__, 'ale_py': ale_py.__version__, 'numpy': np.__version__,
    'gpu': torch.cuda.get_device_name(0),
    'gpu_memory_gib': torch.cuda.get_device_properties(0).total_memory / 2**30,
    'repository': REPO_URL,
    'revision': revision.stdout.strip() if revision.returncode == 0 else 'unknown',
}
print(json.dumps(info, indent=2))
(DIAGNOSTICS_DIR / 'environment.json').write_text(json.dumps(info, indent=2))
smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=True).stdout
print(smi)
(DIAGNOSTICS_DIR / 'nvidia-smi.txt').write_text(smi)


In [ ]:
def run_python(*args, log_name):
    """Show live output and retain a log; raise immediately on command failure."""
    command = [sys.executable, '-u', *map(str, args)]
    print('Running:', ' '.join(command), flush=True)
    with (DIAGNOSTICS_DIR / log_name).open('w') as log:
        process = subprocess.Popen(command, cwd=SOURCE_DIR, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)


## 4. Run regression tests, CUDA smoke training, and resume

Expect the CUDA tests to run, not skip. If a cell fails, stop here and share its output
or the matching file in `diagnostics/`. The smoke run checks execution, not learning.
It uses its own output directory. On repeat visits, a completed smoke budget is a no-op.


In [ ]:
run_python('-m', 'pytest', '-q', '-rs', 'tests', log_name='tests.log')


In [ ]:
SMOKE_DIR = RUNS_DIR / (RUN_NAME + '-smoke')
run_python('ga.py', '-c', 'configurations/ga_smoke_config.json', '-o', SMOKE_DIR,
           '--device', 'cuda', log_name='smoke.log')
run_python('ga.py', '-c', 'configurations/ga_smoke_config.json', '-o', SMOKE_DIR,
           '--device', 'cuda', '--timesteps', 24000, log_name='smoke-resume.log')


## 5. Optional pretrained Frostbite replay

Set `RUN_PRETRAINED = True` in the settings cell to enable this. It decodes a nearly
1 GB noise table and checks the larger network. Scores vary; the local CPU reference
averaged approximately 2,757 over three episodes, not a required GPU score.


In [ ]:
if RUN_PRETRAINED:
    run_python('play.py', '--seeds', 'pretrained/frostbite_uber_elite.json', '--episodes', 3,
               '--device', 'cuda', '--no-video', log_name='pretrained.log')
else:
    print('Optional pretrained replay disabled.')


## 6. Train or resume

This is the longer experiment. Start with the 1-million-step budget and inspect
`TimestepsPerSecondThisIter` and the printed ETA. Tests and validation add runtime;
the budget can overshoot at generation boundaries. A million steps is a pipeline and
throughput trial, not a guarantee of a strong policy.

To continue after reaching the budget, increase `TOTAL_STEPS` (for example to
`50_000_000`) in the settings cell, rerun that cell, then rerun this training cell.
Keep the same `RUN_NAME`. After a runtime replacement, rerun setup first.


In [ ]:
run_python('ga.py', '-c', 'configurations/ga_colab_config.json', '-o', OUT_DIR,
           '--device', 'cuda', '--num-envs', NUM_ENVS, '--timesteps', TOTAL_STEPS,
           log_name='training.log')


## 7. Replay your best policy

The first episode becomes a grayscale video. The command reports scores for three
episodes and keeps the video in Drive.


In [ ]:
from IPython.display import Video, display
run_python('play.py', '--run', OUT_DIR, '--episodes', 3, '--device', 'cuda', log_name='replay.log')
best = json.loads((OUT_DIR / 'best.json').read_text())
display(Video(str(OUT_DIR / f"best_{best['game']}.mp4"), embed=True))


For help with this run, share `environment.json`, the pytest summary, and the first
traceback or last smoke/training table. These are under the printed output directory's
`diagnostics/` folder. Logs there are replaced when their corresponding cell is rerun;
training's `log.txt` and `progress.csv` append across resumes.
